# One way ANOVA

In [0]:
import pandas as pd
# import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
np.random.seed(42)

In [0]:
import numpy as np
import pandas as pd
from scipy import stats

def anova(*args):
    # Number of groups and total observations
    k = len(args)
    N = sum(len(arg) for arg in args)
    
    # Means of each group
    group_means = [np.mean(arg) for arg in args]
    overall_mean = np.mean([x for group in args for x in group])
    
    # Between-group sum of squares (SSB)
    ssb = sum(len(arg) * (np.mean(arg) - overall_mean) ** 2 for arg in args)
    
    # Within-group sum of squares (SSW)
    ssw = sum(sum((x - np.mean(arg)) ** 2 for x in arg) for arg in args)
    
    # Total sum of squares (SST)
    sst = ssb + ssw
    
    # Degrees of freedom
    df_between = k - 1
    df_within = N - k
    
    # Mean squares
    msb = ssb / df_between
    mse = ssw / df_within
    
    # F-statistic and p-value
    F = msb / mse
    p_value = stats.f.sf(F, df_between, df_within)
    
    # Print means
    print("Group Means:")
    for i, m in enumerate(group_means, 1):
        print(f"  Mean of Group {i}: {m:.2f}")
    print(f"\nOverall Mean: {overall_mean:.2f}\n")
    
    # ANOVA summary table
    table = pd.DataFrame({
        "Source": ["Between Groups", "Within Groups", "Total"],
        "SS": [ssb, ssw, sst],
        "df": [df_between, df_within, df_between + df_within],
        "MS": [msb, mse, None],
        "F": [F, None, None],
        "p-value": [p_value, None, None]
    })

    F_upper = stats.f.ppf(0.95, df_between, df_within)
    print(f"F Upper: {F_upper:.2f}")
    if F > F_upper:
        print("Reject the null hypothesis. There is a significant difference between the group means.")
    else:
        print("Fail to reject the null hypothesis. There is no significant difference between the group means.")
    
    return table

# Example Data
result = anova(
    [254, 263, 241, 237, 251],
    [234, 218, 235, 227, 216],
    [200, 222, 197, 206, 204]
)

print(result.round(4))


# Two Way ANOVA

In [0]:
import numpy as np
import pandas as pd
from scipy import stats

def two_way_anova(data):
    """
    data: 2D list or numpy array
    rows = levels of Factor A
    columns = levels of Factor B
    each cell contains multiple observations
    shape = (a, b, n)
    """

    data = np.array(data)
    
    a, b, n = data.shape  # levels of A, B, observations
    
    grand_mean = np.mean(data)
    
    # Means
    mean_A = np.mean(data, axis=(1,2))
    mean_B = np.mean(data, axis=(0,2))
    mean_cell = np.mean(data, axis=2)

    # Sum of Squares
    SSA = b*n * np.sum((mean_A - grand_mean)**2)
    SSB = a*n * np.sum((mean_B - grand_mean)**2)

    SSAB = n * np.sum((mean_cell 
                       - mean_A[:,None] 
                       - mean_B[None,:] 
                       + grand_mean)**2)

    SSE = np.sum((data - mean_cell[:,:,None])**2)

    SST = SSA + SSB + SSAB + SSE

    # Degrees of freedom
    df_A = a - 1
    df_B = b - 1
    df_AB = (a - 1)*(b - 1)
    df_E = a*b*(n - 1)

    # Mean Squares
    MSA = SSA / df_A
    MSB = SSB / df_B
    MSAB = SSAB / df_AB
    MSE = SSE / df_E

    # F statistics
    FA = MSA / MSE
    FB = MSB / MSE
    FAB = MSAB / MSE

    # p-values
    pA = stats.f.sf(FA, df_A, df_E)
    pB = stats.f.sf(FB, df_B, df_E)
    pAB = stats.f.sf(FAB, df_AB, df_E)

    table = pd.DataFrame({
        "Source": ["Factor A", "Factor B", "Interaction", "Error", "Total"],
        "SS": [SSA, SSB, SSAB, SSE, SST],
        "df": [df_A, df_B, df_AB, df_E, df_A+df_B+df_AB+df_E],
        "MS": [MSA, MSB, MSAB, MSE, None],
        "F": [FA, FB, FAB, None, None],
        "p-value": [pA, pB, pAB, None, None]
    })

    return table


# Example Data
# Factor A = Teaching Method (2 levels)
# Factor B = Study Hours (3 levels)
# 3 observations per cell

data = [
    [[85,88,90], [78,80,82], [92,94,96]],   # Method 1
    [[83,85,87], [76,79,81], [89,91,93]]    # Method 2
]

result = two_way_anova(data)
print(result.round(4))

Using a significance level of **α = 0.05**, the interpretations are:

| Source                  | F-value | p-value | Interpretation                                                                                                                                                    |
| ----------------------- | ------: | ------: | ----------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Factor A**            |  5.1279 |  0.0429 | Since **p < 0.05**, Factor A has a **statistically significant effect** on the response variable, indicating that the means differ across the levels of Factor A. |
| **Factor B**            | 54.5000 |  0.0000 | Since **p < 0.05**, Factor B has a **highly significant effect** on the response variable, suggesting strong differences among its groups.                        |
| **Interaction (A × B)** |  0.2442 |  0.7871 | Since **p > 0.05**, there is **no significant interaction effect**, meaning the effect of Factor A does not depend on the levels of Factor B (and vice versa).    |

### Overall Conclusion

> The response variable is significantly influenced by **Factor A and Factor B individually**, but there is **no evidence that the two factors interact with each other** in affecting the outcome.


**Tukey Cramer Procedure**

- Tells which population means are significantly different
- Done after rejection of equal means in ANOVA
- Allows pair-wise comparisons

# Variance, Covariance

- [Covariance and Correlation](https://www.geeksforgeeks.org/data-analysis/mathematics-covariance-and-correlation/)
- [Covariance matrix](https://www.geeksforgeeks.org/maths/covariance-matrix/)

# Correlation, Auto correlation

Correlation quantifies the strength and direction of the relationship between two variables but does not imply causation.

In [0]:
import pandas as pd

df = pd.DataFrame({
    "Hours_Studied": [1,2,3,4,5,6,7,8,9,10],
    "Scores": [50,55,60,30,70,20,80,85,90,95]
})

df.corr()

In [0]:
# Importing the necessary modules
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal


# plt.style.use('seaborn-dark')
plt.rcParams['figure.figsize']=14,6

# Initializing the random seed
random_seed=1000

# List containing the variance
# covariance values
cov_val = [-0.8, 0, 0.3]

# Setting mean of the distributino to
# be at (0,0)
mean = np.array([0,0])

# Iterating over different covariance
# values
for idx, val in enumerate(cov_val):
	plt.subplot(1,3,idx+1)

	# Initializing the covariance matrix
	cov = np.array([[4, val], [val, 1]])

	# Generating a Gaussian bivariate distribution
	# with given mean and covariance matrix
	distr = multivariate_normal(cov = cov, mean = mean,
								seed = random_seed)

	# Generating 5000 samples out of the
	# distribution
	data = distr.rvs(size = 5000)

	# Plotting the generated samples
	plt.plot(data[:,0],data[:,1], 'o', c='lime',
			markeredgewidth = 0.5,
			markeredgecolor = 'black')
	plt.title(f'Covariance between x1 and x2 = {val}')
	plt.xlabel('x1')
	plt.ylabel('x2')
	plt.axis('equal')

plt.show()


## Coorelation Crossover

Correlation is a statistical measure that quantifies the strength and direction of a linear relationship between two variables. In the context of Machine Learning (ML), correlation analysis is a critical step in **Exploratory Data Analysis (EDA)** and **Feature Engineering**.
The most common measure is the **Pearson correlation coefficient** ( or ), which ranges from  to .

In [0]:
> STATS > Part 5
> Time Series Analysis
> Data Manipulation Analysis > Correlation
> Learning Techniques > Linear Regression > Residual Analysis - LINE
> Learning Techniques > Decision Trees > Feature selection

%md
### I. Interpretation of Correlation Between Features


When examining the correlation between two independent features in your dataset:



- Direction: Positive Strength: Strong / Very Strong

As one feature increases, the other tends to increase. This indicates potential **multicollinearity** or **redundancy**. Highly correlated features provide similar information to the model, which can make the model unstable (e.g., in linear models like linear regression) or less interpretable. **Action**: Consider removing one of the highly correlated features or using dimensionality reduction techniques like **Principal Component Analysis (PCA)**.



- Direction: Negative Strength: Strong / Very sTrong

As one feature increases, the other tends to decrease. This also indicates potential **multicollinearity** and redundancy. **Action**: Similar to strong positive correlation, consider removing one.


- Direction: Close to Zero Strength: Weak / None

There is little to no linear relationship between the two features. This is generally preferred for independent features, as it suggests they are providing unique information to the model.


**Key Considerations:**
- Multicollinearity: High correlation between independent features can make the feature importance or coefficients of individual features in linear models unreliable or difficult to interpret, as the model struggles to isolate the individual effect of each feature.
- Redundancy: Highly correlated features are redundant, meaning including both may unnecessarily increase model complexity and training time without adding significant predictive power.


### II. Interpretation from Correlating Features with the Outcome Variable

When examining the correlation between an **independent feature** and the outcome variable (target or dependent variable):


- Direction: Positive Strength: Strong / Very Strong

The feature moves in the same direction as the target variable (e.g., as the feature value increases, the target value tends to increase). This feature is generally highly relevant and likely to be a **strong predictor** for the outcome.



- Direction: Negative Strength: Strong / Very sTrong

The feature moves in the opposite direction of the target variable (e.g., as the feature value increases, the target value tends to decrease). This feature is also generally highly relevant and a **strong predictor**.


- Direction: Close to Zero Strength: Weak / None

There is little to no linear relationship between the feature and the target variable. This suggests the feature may be **irrelevant or a weak linear predictor**. **Action**: Consider dropping these features through _**Feature Selection**_ to simplify the model and potentially improve generalization, unless domain knowledge suggests a non-linear relationship.



**Key Inferences:**
- Feature Relevance: The absolute value of the correlation coefficient (closer to  or ) serves as a preliminary measure of the feature's relevance and predictive power for the target variable.
- Direction of Relationship: The sign of the coefficient indicates the nature of the influence: positive means a direct relationship, and negative means an inverse relationship.
- Feature Selection: Correlation analysis acts as a **filter method** for feature selection. Features with a strong correlation (positive or negative) to the target are typically retained as they possess greater predictive utility.


**Crucial Caveat**: **_Correlation does not imply causation_** . A high correlation simply indicates that two variables tend to co-vary, which may be due to a genuine causal link, a common underlying factor, or pure chance (spurious correlation). In ML, correlation is used for its **predictive power**, regardless of causation.